# Chapter 10 — Recurrent Networks: Sequence Modeling Through Time

**Math for ML Engineers** · companion notebook

This notebook mirrors §10.2 (backpropagation through time) and §10.4 (LSTM gates) of the chapter, building a vanilla RNN and an LSTM from raw NumPy, verifying BPTT gradients against finite differences, and then measuring how gradient magnitude behaves as sequences get longer. **Key takeaway: the vanishing-gradient problem in vanilla RNNs is not a minor numerical nuisance — it is the single architectural limitation that made attention-based models the default choice for long sequences.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(10)

## Setup: the vanilla RNN recurrence

A vanilla RNN maintains a hidden state $h_t \in \mathbb{R}^{d_h}$ that summarizes everything seen up to time $t$. At each step it mixes the current input $x_t$ with the previous hidden state through a single $\tanh$ nonlinearity (§10.3):

$$
h_t = \tanh\!\big(W_x x_t + W_h h_{t-1} + b\big), \qquad y_t = W_y h_t + b_y
$$

Training this recurrence with backpropagation through time (BPTT) requires differentiating the loss

$$
\mathcal{L} = \frac{1}{T} \sum_{t=1}^{T} \frac{1}{2} \lVert y_t - \hat{y}_t \rVert^2
$$

back through *every* time step. Because $h_t$ depends on $h_{t-1}$, the chain rule produces a product of Jacobians:

$$
\frac{\partial \mathcal{L}}{\partial h_1} \;\propto\; \prod_{t=2}^{T} \frac{\partial h_t}{\partial h_{t-1}} = \prod_{t=2}^{T} \operatorname{diag}\!\big(1 - h_t^2\big)\, W_h
$$

Each factor has spectral norm bounded by $\lVert W_h \rVert$ times the largest value of $1 - h_t^2 \le 1$ (with equality only where $h_t = 0$, i.e. inside $\tanh$'s linear regime). Multiply $T$ such factors together and the product either shrinks toward zero or blows up — the gradient must survive $T$ sequential contractions to reach the earliest time step.

In [ ]:
def rnn_step(x_t, h_prev, Wx, Wh, b):
    """h_t = tanh(Wx x_t + Wh h_{t-1} + b)."""
    return np.tanh(Wx @ x_t + Wh @ h_prev + b)


def rnn_forward(X, h0, Wx, Wh, b, Wy, by):
    T, _ = X.shape
    d_h  = h0.shape[0]
    H    = np.zeros((T, d_h))
    h    = h0.copy()
    for t in range(T):
        h     = rnn_step(X[t], h, Wx, Wh, b)
        H[t]  = h
    Y = H @ Wy.T + by
    return H, Y


# Small, well-conditioned problem: 3-dim input, 4-dim hidden state, 5 time steps
d_x, d_h, d_y, T = 3, 4, 2, 5

Wx = np.random.randn(d_h, d_x) * 0.5
Wh = np.random.randn(d_h, d_h) * 0.5
b  = np.random.randn(d_h) * 0.1
Wy = np.random.randn(d_y, d_h) * 0.5
by = np.random.randn(d_y) * 0.1

X = np.random.randn(T, d_x)
h0 = np.zeros(d_h)
Y_target = np.random.randn(T, d_y)

H, Y = rnn_forward(X, h0, Wx, Wh, b, Wy, by)
print("H shape:", H.shape, " Y shape:", Y.shape)

## BPTT: accumulating gradients across time steps

The implementation below walks time **backward**, accumulating gradients into `dWx`, `dWh`, `db` at every step rather than overwriting them — because the *same* weight matrices $W_x, W_h, b$ are reused at every time step, so their total gradient is a sum over $t$ of the per-step contribution.

The key recursive quantity is `dh_next`: the gradient flowing in from the *future* (from $h_{t+1}$'s dependence on $h_t$), added to the gradient arriving from the current step's output $y_t$. This running `dh` is exactly the "gradient signal" whose norm we'll track as a function of sequence length in the next section — it's the same quantity that shrinks or survives as it's pushed through repeated $\tanh' \cdot W_h$ multiplications.

We validate the analytic BPTT implementation against numerical (finite-difference) gradients on $W_h$ — if the two agree to machine precision, the backward pass is provably correct, not just "looks plausible."

In [ ]:
def rnn_bptt(X, Y_target, H, h0, Wx, Wh, b, Wy, by):
    """Backpropagation through time. Loss = (1/T) sum_t ||y_t - target_t||^2 / 2."""
    T, d_x = X.shape
    d_h    = h0.shape[0]
    H_full = np.vstack([h0[np.newaxis, :], H])

    Y   = H @ Wy.T + by
    dY  = (Y - Y_target) / T   # d/dY of (1/T) sum_t (1/2)||y_t - target_t||^2

    dWy = dY.T @ H
    dby = dY.sum(axis=0)

    dWx = np.zeros_like(Wx)
    dWh = np.zeros_like(Wh)
    db  = np.zeros_like(b)
    dh_next = np.zeros(d_h)

    for t in reversed(range(T)):
        dh    = Wy.T @ dY[t] + dh_next
        dtanh = (1 - H[t] ** 2) * dh
        db   += dtanh
        dWx  += np.outer(dtanh, X[t])
        dWh  += np.outer(dtanh, H_full[t])
        dh_next = Wh.T @ dtanh

    return dWx, dWh, db, dWy, dby


def loss_fn(X, Y_target, h0, Wx, Wh, b, Wy, by):
    H, Y = rnn_forward(X, h0, Wx, Wh, b, Wy, by)
    return 0.5 * np.mean(np.sum((Y - Y_target) ** 2, axis=1))


dWx, dWh, db, dWy, dby = rnn_bptt(X, Y_target, H, h0, Wx, Wh, b, Wy, by)

# Finite-difference check on Wh (the recurrent weight matrix -- the one that
# matters most for the vanishing-gradient story below)
eps = 1e-5
dWh_numeric = np.zeros_like(Wh)
for i in range(Wh.shape[0]):
    for j in range(Wh.shape[1]):
        Wh_plus, Wh_minus = Wh.copy(), Wh.copy()
        Wh_plus[i, j]  += eps
        Wh_minus[i, j] -= eps
        L_plus  = loss_fn(X, Y_target, h0, Wx, Wh_plus,  b, Wy, by)
        L_minus = loss_fn(X, Y_target, h0, Wx, Wh_minus, b, Wy, by)
        dWh_numeric[i, j] = (L_plus - L_minus) / (2 * eps)

max_err = np.max(np.abs(dWh - dWh_numeric))
print(f"max |analytic - numeric| gradient error on Wh: {max_err:.3e}")

## The main event: how far can a gradient travel?

The gradient check above confirms BPTT is implemented correctly for *one* fixed sequence length. The question that actually matters in practice is: **what happens to that gradient as $T$ grows?**

We'll measure the norm of the gradient signal reaching the *first* time step, $\lVert \partial \mathcal{L} / \partial h_1 \rVert$, as a function of sequence length $T$, for two architectures on matched random weights:

- **Vanilla RNN** — the gradient must pass through $T$ multiplications by $\operatorname{diag}(1-h_t^2) W_h$. With $\tanh$ saturating (its derivative $\le 1$ and often $\ll 1$ once $|h_t|$ grows), this product tends to shrink geometrically with $T$.
- **LSTM** — we'll add the corresponding backward pass through the cell state $c_t = f_t \odot c_{t-1} + i_t \odot g_t$ (§10.4). Because this update is **additive** (not repeatedly squashed through $\tanh$), the gradient $\partial \mathcal{L}/\partial c_1$ only picks up a multiplicative forget-gate factor $f_t \in (0,1)$ per step rather than a full Jacobian contraction. We also initialize the forget-gate bias positively (a standard practical trick, biasing gates toward "remember" at the start of training) — combined with the additive path, this produces a much gentler decay than the vanilla RNN's repeated $\tanh$-Jacobian contractions.

The plot below is the single most important figure in this chapter: it turns the abstract "vanishing gradient" claim into a concrete, measured curve.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -30, 30)))


def lstm_step(x_t, h_prev, c_prev, Wf, Wi, Wg, Wo, bf, bi, bg, bo):
    """One LSTM step. h_t = o_t * tanh(c_t)."""
    v = np.concatenate([h_prev, x_t])
    f = sigmoid(Wf @ v + bf)
    i = sigmoid(Wi @ v + bi)
    g = np.tanh(Wg @ v + bg)
    o = sigmoid(Wo @ v + bo)
    c = f * c_prev + i * g
    h = o * np.tanh(c)
    return h, c, (f, i, g, o, c, c_prev)


def lstm_forward(X, h0, c0, Wf, Wi, Wg, Wo, bf, bi, bg, bo):
    T = len(X)
    d_h = h0.shape[0]
    H, C = np.zeros((T, d_h)), np.zeros((T, d_h))
    h, c = h0.copy(), c0.copy()
    cache = []
    for t in range(T):
        h, c, step_cache = lstm_step(X[t], h, c, Wf, Wi, Wg, Wo, bf, bi, bg, bo)
        H[t], C[t] = h, c
        cache.append(step_cache)
    return H, C, cache


def rnn_grad_norm_at_start(T, d_x=3, d_h=4, seed=0):
    """||dL/dh_1|| for a vanilla RNN run for T steps, backprop from a loss at h_T."""
    rng = np.random.RandomState(seed)
    Wx = rng.randn(d_h, d_x) * 0.5
    Wh = rng.randn(d_h, d_h) * 0.5
    b  = rng.randn(d_h) * 0.1
    X  = rng.randn(T, d_x)
    h0 = np.zeros(d_h)

    H, _ = rnn_forward(X, h0, Wx, Wh, b, np.eye(d_h), np.zeros(d_h))
    H_full = np.vstack([h0[np.newaxis, :], H])

    # Seed the backward pass with a unit gradient at the final hidden state,
    # exactly as if the loss only depended on h_T.
    dh_next = np.ones(d_h)
    for t in reversed(range(T)):
        dtanh = (1 - H[t] ** 2) * dh_next
        dh_next = Wh.T @ dtanh
    return np.linalg.norm(dh_next)


def lstm_grad_norm_at_start(T, d_x=3, d_h=4, seed=0, forget_bias=2.0):
    """||dL/dc_1|| for an LSTM run for T steps, backprop from a loss at c_T.

    Only the forget-gate path through the cell state is modeled here (the
    dominant term for long-range credit assignment) -- gradient into c_{t-1}
    picks up a factor of f_t, not a full tanh-Jacobian contraction. The
    forget-gate bias is initialized positive (a standard LSTM practice, e.g.
    Jozefowicz et al. 2015) so gates start biased toward "remember" -- this
    is what actually gives the additive path its advantage in practice, not
    just the additive structure alone.
    """
    rng = np.random.RandomState(seed)
    v_dim = d_h + d_x
    Wf = rng.randn(d_h, v_dim) * 0.5
    Wi = rng.randn(d_h, v_dim) * 0.5
    Wg = rng.randn(d_h, v_dim) * 0.5
    Wo = rng.randn(d_h, v_dim) * 0.5
    bf = rng.randn(d_h) * 0.1 + forget_bias
    bi = rng.randn(d_h) * 0.1
    bg = rng.randn(d_h) * 0.1
    bo = rng.randn(d_h) * 0.1
    X  = rng.randn(T, d_x)
    h0, c0 = np.zeros(d_h), np.zeros(d_h)

    _, _, cache = lstm_forward(X, h0, c0, Wf, Wi, Wg, Wo, bf, bi, bg, bo)

    dc_next = np.ones(d_h)
    for t in reversed(range(T)):
        f, i, g, o, c, c_prev = cache[t]
        dc_next = f * dc_next  # additive cell-state path: multiply by forget gate only
    return np.linalg.norm(dc_next)


T_values = np.arange(1, 51)
rnn_norms  = np.array([rnn_grad_norm_at_start(T)  for T in T_values])
lstm_norms = np.array([lstm_grad_norm_at_start(T) for T in T_values])

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(T_values, rnn_norms,  'o-', label='Vanilla RNN', color='crimson', markersize=4)
ax.semilogy(T_values, lstm_norms, 's-', label='LSTM (cell-state path)', color='steelblue', markersize=4)
ax.set_xlabel('Sequence length T')
ax.set_ylabel(r'$\|\partial \mathcal{L} / \partial h_1\|$ (log scale)')
ax.set_title('Gradient norm reaching the first time step vs. sequence length')
ax.legend()
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print(f"Vanilla RNN gradient norm at T=5:  {rnn_norms[4]:.3e}")
print(f"Vanilla RNN gradient norm at T=50: {rnn_norms[49]:.3e}")
print(f"LSTM       gradient norm at T=5:   {lstm_norms[4]:.3e}")
print(f"LSTM       gradient norm at T=50:  {lstm_norms[49]:.3e}")
print(f"Ratio of decay (RNN T=50 / T=5):  {rnn_norms[49] / rnn_norms[4]:.3e}")
print(f"Ratio of decay (LSTM T=50 / T=5): {lstm_norms[49] / lstm_norms[4]:.3e}")

## Why this exact problem killed RNNs in production

The curve above is not a toy artifact — it is *the* reason the field moved away from recurrent architectures for long-sequence modeling. In a vanilla RNN, the path length between time step 1 and time step $T$ is $O(T)$: information (and gradient) must pass through $T$ sequential $\tanh$ nonlinearities and multiplications by $W_h$. Every one of those steps is a chance for the signal to shrink (vanishing gradients, the common case when weights are modest) or blow up (exploding gradients, mitigated in practice by gradient clipping).

This is precisely the limitation that motivated **self-attention**. In a Transformer, every position can attend directly to every other position in a single layer — the path length between any two tokens, however far apart, is $O(1)$, not $O(T)$. There is no chain of $T$ nonlinear contractions for the gradient to survive. This is a first-order reason (alongside parallelizability across the sequence dimension) that attention-based architectures displaced RNNs/LSTMs as the default for language modeling, translation, and virtually every long-context sequence task in production systems today.

Notice too what made the LSTM better than the vanilla RNN *before* attention existed: the additive cell-state update $c_t = f_t \odot c_{t-1} + i_t \odot g_t$ creates a path where the gradient is multiplied by a forget gate $f_t \in (0, 1)$ instead of being pushed through a full $\tanh$-Jacobian-times-$W_h$ contraction at every step. That additive "gradient superhighway" is architecturally the same trick as the **residual/skip connections** that make very deep ResNets and Transformer residual streams trainable: $x_{l+1} = x_l + F(x_l)$ lets gradient flow through the identity term unimpeded, bypassing the nonlinear branch entirely. Same fix — an additive bypass around a squashing nonlinearity — applied at three different points in the history of deep learning.

**Key takeaway: vanishing gradients are a consequence of gradient path length, and every major architectural advance since vanilla RNNs — LSTM gating, residual connections, self-attention — is fundamentally a way of shortening or bypassing that path.**

In [ ]:
# Quantify path length directly: how many multiplicative "hops" does the
# gradient survive before falling below a floor, for RNN vs LSTM vs the O(1)
# path length of self-attention?

floor = 1e-3   # treat the gradient as "vanished" below this norm
init_norm = 1.0

def hops_to_vanish(norms, T_values, floor):
    below = np.where(norms < floor)[0]
    return T_values[below[0]] if len(below) > 0 else None

rnn_vanish_T  = hops_to_vanish(rnn_norms,  T_values, floor)
lstm_vanish_T = hops_to_vanish(lstm_norms, T_values, floor)

print(f"Vanilla RNN gradient drops below {floor:.0e} at T = {rnn_vanish_T}")
print(f"LSTM gradient drops below {floor:.0e} at T = "
      f"{'not reached within T<=50' if lstm_vanish_T is None else lstm_vanish_T}")

# Path length comparison: RNN/LSTM path length grows with T; self-attention's
# does not, by construction (every pair of positions is one hop apart).
print()
print(f"{'Sequence length T':>18} | {'RNN/LSTM path length':>22} | {'Self-attention path length':>27}")
for T in [10, 100, 1000, 10000]:
    print(f"{T:>18} | {T:>22} | {1:>27}")

print()
print("Path length is O(T) for recurrence, O(1) for self-attention -- independent of T.")
print("This is the structural reason attention scales to long contexts where RNNs struggle.")

## Exercise

1. The forward-mode analysis above tracks $\lVert \partial \mathcal{L}/\partial h_1 \rVert$ assuming the *forget gate* $f_t$ is the only multiplicative factor along the LSTM's cell-state path. Extend `lstm_grad_norm_at_start` to also account for the gradient path through the hidden state $h_t = o_t \odot \tanh(c_t)$ (i.e. include the $\tanh'(c_t)$ and output-gate terms), and re-plot both curves. Does the extra path change the qualitative conclusion? See §10.4 for the exact one-step LSTM Jacobian, including the terms beyond the forget gate.

2. Derive, analogous to the $\tanh$-Jacobian bound used in §10.3, an upper bound on $\lVert \partial h_t / \partial h_{t-1} \rVert$ for a residual-connected recurrent update $h_t = h_{t-1} + \tanh(W_x x_t + W_h h_{t-1} + b)$ (i.e. adding a skip connection directly to the vanilla RNN recurrence). Show that this bound no longer forces the Jacobian's spectral norm below 1, and explain why that matches the "gradient superhighway" argument made above. See §8.6 for the general treatment of residual connections and Jacobian conditioning.